In [5]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))

In [2]:
import copy
import optuna
import numpy as np
from transformers import AutoTokenizer

from src.train import train_single_fold, tokenize_df
from src.config import load_config
from src.data_holdout import get_pooled_df, get_fold_from_disk

import yaml
import torch
from datasets import Dataset
from pyprojroot import here
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from sklearn.metrics import f1_score

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_NAME = "microsoft/deberta-v3-base"

In [5]:
#check appropriately frozen

def freeze_base(model):
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False
            
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)
freeze_base(model)

trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
print(trainable_names)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 3172.74it/s, Materializing param=deberta.encoder.rel_embeddings.weight]                     
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.bias                         | MI

['classifier.weight', 'classifier.bias']
Trainable: 2,307 / 184,424,451 (0.0013%)


In [ ]:
#open shared settings
with open(here("configs/base.yaml"), "r") as f:
    base_data = yaml.full_load(f)

#get df
full_df = get_pooled_df()

# Same pattern as collaborator: all folds
train_folds, val_folds = get_fold_from_disk(full_df)

#load tokenizer and write tokenize function
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fold(df):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=base_data["max_length"],
        )

    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

#tokenize all folds
train_ds = [tokenize_fold(df) for df in train_folds]
val_ds = [tokenize_fold(df) for df in val_folds]

for i in range(len(train_ds)):
    print(f"Fold {i}: train={len(train_ds[i])} val={len(val_ds[i])}")

# define metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

#Freeze encoder, train classifier head only
def freeze_base(model):
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False

#Set up Optuna objective
def objective(trial):
    # Match collaborator's structure, but use a frozen-head-appropriate LR range
    lr = trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True) #higher LR for classification head only
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.05, 0.2) #empirically found that higher is better for classification head only
    num_epochs = trial.suggest_int("num_train_epochs", 12, 30) #not as many epochs needed for classification head only

    f1_scores = []

    for fold in range(5):
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=3
        )
        freeze_base(model)

        training_args = TrainingArguments(
            output_dir=str(here(f"results/optuna/deberta_frozen/trial-{trial.number}/fold-{fold}")),
            eval_strategy=base_data["eval_strategy"],
            save_strategy=base_data["save_strategy"],
            learning_rate=lr,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            num_train_epochs=num_epochs,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=base_data["greater_is_better"],
            fp16=False,
            seed=base_data["seed"],
            logging_steps=10,
            report_to=base_data["report_to"],
            save_total_limit=1,
            skip_memory_metrics=True,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds[fold],
            eval_dataset=val_ds[fold],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )

        trainer.train()
        f1_scores.append(trainer.evaluate()["eval_macro_f1"])

        del model
        torch.cuda.empty_cache()

    return float(np.mean(f1_scores))

# Study definition
db_path = here("results/optuna/optuna_study_deberta_frozen.db")
storage_url = f"sqlite:///{db_path}"
n_startup_trials = 5

print(storage_url)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=n_startup_trials,
        seed=42,
    ),
    study_name="deberta-frozen-hyperparameter-search",
    storage=storage_url,
    load_if_exists = True
)

study.optimize(objective, n_trials=50)

# Best trials
print(f"\n{'='*60}")
print(f"  Best macro_f1: {study.best_trial.value:.4f}")
print(f"  Best params:   {study.best_trial.params}")
print(f"{'='*60}")


/workspace/exaggeration-detection/src/data_holdout.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(ENCODED_LABELS)
Map: 100%|██████████| 106/106 [00:00<00:00, 8058.22 examples/s]


Fold 0: train=424 val=106
Fold 1: train=424 val=106
Fold 2: train=424 val=106
Fold 3: train=424 val=106
Fold 4: train=424 val=106
sqlite:////workspace/exaggeration-detection/results/optuna/optuna_study_deberta_frozen.db


[I 2026-03-22 17:18:24,299] A new study created in RDB with name: deberta-frozen-hyperparameter-search
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 2416.31it/s, Materializing param=deberta.encoder.rel_embeddings.weight]                     
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED 

ValueError: Attempting to unscale FP16 gradients.

In [7]:
print(os.getcwd())

/workspace/exaggeration-detection/notebooks


In [8]:
import optuna

study = optuna.load_study(
    study_name="pubmedbert_frozen",
    storage="sqlite:////home/rebecca_m/mids266-exaggeration/results/optuna_study_pubmedbert_frozen.db"
)

print(study.best_trial)
print(study.best_value)
print(study.best_params)

OperationalError: (sqlite3.OperationalError) unable to open database file
(Background on this error at: https://sqlalche.me/e/20/e3q8)